# Anchored PC on MNIST: short paired mechanism pilot

Standalone float64 JAX MLP 784 → 128 → 64 → 10, tanh, plain SGD, no dropout or weight decay. Use a CPU runtime. Train on 10,000 images and tune on 5,000 disjoint validation images from the standard 60,000-image training split. Do not use test images for tuning.

Verify the spike-only BP anchor and local energy derivatives, then calibrate PC at initialization and after 20 BP updates. Training convergence limits are residual ratio 0.001 and T/2T gradient change 1%; calibration uses stricter safety margins. Maximum inference is 480 steps. The pilot has a 30-minute budget and a hard 31-minute process timeout.

Eight methods, equal three-point LR search on seed 0 for 50 updates, then three fresh paired seeds for 200 updates. This is a short validation pilot, not evidence about BERT, forgetting, novelty or final MNIST test accuracy. Upload the two Python files from this kit when prompted.


In [ ]:
from pathlib import Path
from google.colab import files
import os, sys, subprocess, hashlib, json
if any(not Path(name).exists() for name in ('anchored_mnist.py', 'test_anchored_mnist.py')):
    uploaded = files.upload()
expected = {'anchored_mnist.py': 'c96bcd91d129b0d37b1804369ec741d77eb006d2192fa9f6427c66e6df4fa31c', 'test_anchored_mnist.py': '562200b7a4ac3d39e36f8455f0a3716455c445a734002a7975475ba7de9ff17e'}
for name, digest in expected.items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == digest, name
os.environ['JAX_PLATFORMS'] = 'cpu'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'jax==0.11.2', 'numpy==2.5.4', 'pytest', 'matplotlib'], check=True)


In [ ]:
# Adapt only the import path for the standalone source file; test bodies stay unchanged.
s = Path('test_anchored_mnist.py').read_text()
old = 'from fabricpc.experiments.anchored_mnist import ('
assert s.count(old) == 1
Path('test_anchored_mnist_standalone.py').write_text(s.replace(old, 'from anchored_mnist import ('))
checked = subprocess.run([sys.executable, '-m', 'pytest', 'test_anchored_mnist_standalone.py', '-q', '-p', 'no:cacheprovider'], capture_output=True, text=True, timeout=180)
print(checked.stdout, checked.stderr)
checked.check_returncode()


In [ ]:
# Public MNIST mirror and checksums are those listed in torchvision/datasets/mnist.py.
from urllib.request import urlretrieve
raw = Path('/content/mnist_raw')
raw.mkdir(exist_ok=True)
resources = {
    'train-images-idx3-ubyte.gz': 'f68b3c2dcbeaaa9fbdd348bbdeb94873',
    'train-labels-idx1-ubyte.gz': 'd53e105ee54ea40749a09fcbcd1e9432',
    't10k-images-idx3-ubyte.gz': '9fb629c4189551a2d022fa330f9573f3',
    't10k-labels-idx1-ubyte.gz': 'ec29112dd5afa0611ce80d1b7f02629c',
}
for name, digest in resources.items():
    path = raw / name
    if not path.exists():
        urlretrieve('https://ossci-datasets.s3.amazonaws.com/mnist/' + name, path)
    assert hashlib.md5(path.read_bytes()).hexdigest() == digest, name
print('All four MNIST files verified')


In [ ]:
import datetime, threading
out = Path('/content') / ('anchored_mnist_' + datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
command = [sys.executable, '-u', 'anchored_mnist.py', '--data-dir', str(raw), '--output-dir', str(out), '--steps', '200', '--tune-steps', '50', '--budget-seconds', '1800']
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
timer = threading.Timer(1860, process.kill)
timer.daemon = True
timer.start()
try:
    with Path('/content/mnist_process.log').open('w') as log:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
    status = process.wait()
finally:
    timer.cancel()
print('Process exit code:', status)
if (out / 'summary.json').exists():
    summary = json.loads((out / 'summary.json').read_text())
    print({k: summary.get(k) for k in ('status', 'solver', 'all_methods_completed', 'elapsed_seconds')})
assert status == 0, 'Inspect saved evidence; a zero exit code alone is not a scientific pass'


In [ ]:
import shutil
shutil.copy('anchored_mnist.py', out / 'source.py')
shutil.copy('/content/mnist_process.log', out / 'process.log')
(out / 'tests.log').write_text(checked.stdout + checked.stderr)
(out / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
evidence = shutil.make_archive(str(out), 'zip', out)
print('Saved evidence:', evidence)
files.download(evidence)
